# Make Semantic Clusters of laboratory tests names

In [ ]:

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering


## Load the initial data we produced 

In [ ]:
# Load saved Parquet files

# target_protein = None
# target_protein = "hba1c"
target_protein = "insulin"

if target_protein is None:
    file_long = "data\\aou_labs_long.parquet"
    file_wide = "data\\aou_labs_wide.parquet"
else:
    # file_long = f"data\\aou_{target_protein}_labs_long.parquet"
    # file_wide = f"data\\aou_{target_protein}_labs_wide.parquet"
    file_long = f"data\\aou_{target_protein}_labs_long_cleaned.parquet"
    file_wide = f"data\\aou_{target_protein}_labs_wide_cleaned.parquet"
    

print("Loading saved Parquet files...")
# df_long = pd.read_parquet(file_long, engine='pyarrow')
df_wide = pd.read_parquet(file_wide, engine='pyarrow')

# print(f"\n Loaded df_long: {df_long.shape[0]:,} rows x {df_long.shape[1]} columns")
print(f" Loaded df_wide: {df_wide.shape[0]:,} rows x {df_wide.shape[1]} columns")

df_wide.head()

In [ ]:


# df_wide["Invasive Systolic Blood Pressure"]
# df_wide["Invasive Systolic Blood Pressure"].describe()


In [ ]:
# df_long[df_long["laboratory_test_name"]=="Invasive Systolic Blood Pressure"]
# df_long[df_long["laboratory_test_name"]=="Invasive Systolic Blood Pressure"].describe()


In [ ]:
df_wide.columns[0:] # do not use the target columns

In [ ]:

target_proteins = ['Glucose', 'HbA1c', 'Insulin', 'Adiponectin', 'Leptin', 'IGFBP1', 'IGFBP2', 'Resistin', 'Glucagon', 'GDF15']

present_targets = [col for col in target_proteins if col in df_wide.columns]

start_column=-1
for prot_name in present_targets:
    if start_column <= list(df_wide.columns[0:]).index(prot_name):
        start_column = list(df_wide.columns[0:]).index(prot_name) +1
    
print(f"start_column = {start_column}")



In [ ]:
# # 1. Read input column names
# input_file = "columns names.txt"
# columns_df = pd.read_table(input_file, header=None, names=["column_name"])

# if target_protein is None:
#     start_column = 10
# if target_protein == "hba1c":
#     # start_column = 5
#     start_column = 8
# if target_protein == "insulin":
#     # start_column = 10
#     start_column = 8

# Clean inputs and strip whitespace
# columns_list = columns_df["column_name"].dropna().astype(str).str.strip().tolist()
columns_list = df_wide.columns[start_column:].dropna().astype(str).str.strip().tolist() # do not use the target columns
print(f"Loaded {len(columns_list)} lab column names.")

## Get the semantic encoder from HuggingFace 🤗

In [ ]:
# 2. Load SapBERT model locally
print("\nLoading SapBERT model (cambridgeltl/SapBERT-from-PubMedBERT-fulltext)...")
model = SentenceTransformer("cambridgeltl/SapBERT-from-PubMedBERT-fulltext")

# # 2. Load MedEmbed model locally
# print("\nLoading MedEmbed model (abhinand/MedEmbed-large-v0.1)...")
# model = SentenceTransformer("abhinand/MedEmbed-large-v0.1")

## Generate semantic embeddings for laboratory tests names

In [ ]:
# 3. Generate embeddings locally
print("Generating embeddings for lab columns...")
embeddings = model.encode(
    columns_list,
    batch_size=64,             # Increase batch size if running on a GPU
    show_progress_bar=True,
    convert_to_numpy=True
)

## Make semantic clusters of the laboratory tests names

In [ ]:
# 4. Semantic Clustering
# Hierarchical Agglomerative Clustering using Cosine Distance
# Lower threshold = stricter/tighter groups | Higher threshold = broader groups
clustering_model = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=0.30,  # Recommended starting point for SapBERT (0.25 - 0.35)
    metric="cosine",
    linkage="average"
)

cluster_labels = clustering_model.fit_predict(embeddings)

## Format and Save the clustering results

### format clusters data

In [ ]:
# 5. Format and Export Results
result_df = pd.DataFrame({
    "column_name": columns_list,
    "cluster_id": cluster_labels
})

for inx in result_df.index:
    result_df.loc[inx,"sample_size"] = len(df_wide[result_df.loc[inx,"column_name"]].dropna())

### print clusters

In [ ]:
# Sort by cluster_id for readability
result_df = result_df.sort_values(by=["cluster_id", "sample_size", "column_name"], ascending=[True, False, True]).reset_index(drop=True)

# Display a preview of identified clusters
print("\n" + "=" * 50)
print(f"Total Clusters Identified: {result_df['cluster_id'].nunique()}")
print("=" * 50)

for cluster_id, group in result_df.groupby("cluster_id"):
    items = group["column_name"].tolist()
    print(f"\n--- Cluster {cluster_id} ({len(items)} items) ---")
    for item in items[:50]:
        print(f"  • {item}")
    if len(items) > 50:
        print(f"  ... and {len(items) - 50} more")

### save clusters data

In [ ]:
# Save output to CSV
if target_protein is None:
    output_filename = "data\\clustered_lab_columns_sapbert.csv"
else:
    output_filename = f"data\\clustered_{target_protein}_lab_columns_sapbert.csv"
result_df.to_csv(output_filename, index=False)
print(f"\nResults saved to '{output_filename}'.")

In [ ]:
result_df

### one cluster

In [ ]:
inx=0
# inx=1079


df_wide[result_df.loc[inx,"column_name"]]
df_wide[result_df.loc[inx,"column_name"]].describe()
# len(df_wide[result_df.loc[inx,"column_name"]].dropna())